In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [2]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

'Skip during test\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Intermediate")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Proximal")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Distal")\nsection_folders = []\nfor folder in series_folder.iterdir():  # Iterate over all items in the folder\n    if folder.is_dir() and folder.name.startswith("S_"):\n        print(f"fould series folder: {folder.name}")\n        section_folders.append(folder)\n'

In [3]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [4]:
series_folder = Path(r".\data\test-series_2")

#TODO: important change this to metadata readout
pixel_size = {
    'Value': 0.010,
    'Axial': 0.07,
    'Unit': 'µm'
}
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

fould series image: stitched_image_S_001_883976504.tiff
fould series image: stitched_image_S_002_192360474.tiff
fould series image: stitched_image_S_003_117038215.tiff
fould series image: stitched_image_S_004_416492709.tiff
fould series image: stitched_image_S_005_471748856.tiff
fould series image: stitched_image_S_006_997114128.tiff
fould series image: stitched_image_S_007_1430161748.tiff
fould series image: stitched_image_S_008_1605682347.tiff
fould series image: stitched_image_S_009_757510793.tiff
fould series image: stitched_image_S_011_424707932.tiff
fould series image: stitched_image_S_012_591550158.tiff
fould series image: stitched_image_S_013_327577099.tiff
fould series image: stitched_image_S_014_2126124089.tiff
fould series image: stitched_image_S_015_1574980897.tiff
fould series image: stitched_image_S_016_2050706348.tiff
fould series image: stitched_image_S_017_999665249.tiff
fould series image: stitched_image_S_018_1457696231.tiff
fould series image: stitched_image_S_019_1

In [5]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)

In [6]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif

In [7]:
"""SKIP during test
#TODO: important change this to metadata readout
pixel_size = {
    'Value': get_pixel_size_from_tif(tif_list_sorted[0].name, tif_list_sorted[0].parent),
    'Axial': 0.07,
    'Unit': 'µm'
}
"""

""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""
print("work on pixel size")

work on pixel size


In [8]:
from atlas.io import create_empty_folder, rm_tree
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment

In [9]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10
tif_list_sorted = sorted(tif_list, key=extract_s_number)

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df.to_csv(output_path.joinpath("z_alignment_results.csv"), index=False)

z_align_df 

Created new (or emptied) folder: data\test-series_2\alignment_results
Processing alignment: ref -> stitched_image_S_001_883976504.tiff, moving -> stitched_image_S_001_883976504.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_001_883976504.tiff, moving -> stitched_image_S_002_192360474.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [-1. -4.]
current shift: [-1. -4.]
Processing alignment: ref -> stitched_image_S_002_192360474.tiff, moving -> stitched_image_S_003_117038215.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [-10.   2.]
current shift: [-10.   2.]
Processing alignment: ref -> stitched_image_S_003_117038215.tiff, moving -> stitched_image_S_004_416492709.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [ 6. -8.]
current shift: [ 6. -8.]
Processing alignment: ref -> stitched_image_S_004_

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,data\test-series_2\stitched_image_S_001_883976...,data\test-series_2\stitched_image_S_001_883976...,"(257, 1208, 96, 703)","(257, 1208, 96, 703)","[0.0, 0.0]","(257, 1208, 96, 703)","[0.0, 0.0]",10
1,data\test-series_2\stitched_image_S_002_192360...,data\test-series_2\stitched_image_S_001_883976...,"(257, 1213, 96, 703)","(257, 1208, 96, 703)","[-1.0, -4.0]","(253, 1209, 95, 702)","[-1.0, -4.0]",10
2,data\test-series_2\stitched_image_S_003_117038...,data\test-series_2\stitched_image_S_002_192360...,"(257, 1206, 96, 703)","(257, 1213, 96, 703)","[-10.0, 2.0]","(255, 1204, 85, 692)","[-11.0, -2.0]",10
3,data\test-series_2\stitched_image_S_004_416492...,data\test-series_2\stitched_image_S_003_117038...,"(257, 1205, 96, 703)","(257, 1206, 96, 703)","[6.0, -8.0]","(247, 1195, 91, 698)","[-5.0, -10.0]",10
4,data\test-series_2\stitched_image_S_005_471748...,data\test-series_2\stitched_image_S_004_416492...,"(372, 1326, 155, 762)","(257, 1205, 96, 703)","[-63.0, -118.0]","(244, 1198, 87, 694)","[-68.0, -128.0]",10
5,data\test-series_2\stitched_image_S_006_997114...,data\test-series_2\stitched_image_S_005_471748...,"(257, 1204, 96, 703)","(372, 1326, 155, 762)","[59.0, 127.0]","(256, 1203, 87, 694)","[-9.0, -1.0]",10
6,data\test-series_2\stitched_image_S_007_143016...,data\test-series_2\stitched_image_S_006_997114...,"(257, 1203, 96, 703)","(257, 1204, 96, 703)","[6.0, -4.0]","(252, 1198, 93, 700)","[-3.0, -5.0]",10
7,data\test-series_2\stitched_image_S_008_160568...,data\test-series_2\stitched_image_S_007_143016...,"(257, 1206, 96, 703)","(257, 1203, 96, 703)","[-12.0, -12.0]","(240, 1189, 81, 688)","[-15.0, -17.0]",10
8,data\test-series_2\stitched_image_S_009_757510...,data\test-series_2\stitched_image_S_008_160568...,"(372, 1322, 156, 762)","(257, 1206, 96, 703)","[-112.0, -71.0]","(284, 1234, 29, 635)","[-127.0, -88.0]",10
9,data\test-series_2\stitched_image_S_011_424707...,data\test-series_2\stitched_image_S_009_757510...,"(257, 1214, 96, 703)","(372, 1322, 156, 762)","[120.0, 89.0]","(258, 1215, 89, 696)","[-7.0, 1.0]",10


In [17]:
# drop sections that are out of place, I need a better procedure for this, in this case easy because they were at the end
# Drop the last 2 rows in-place
#z_align_df.drop(z_align_df.index[-2:], inplace=True)
# TODO: work on a code to drop based on index values, however this might require re-run of alignment.
# a solution would be to uncouple the pairwise alignment from the comulative one, so this would not require
# a full rerun but just a couple of extra operations

In [18]:
from atlas.io import apply_alignment

use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

we will downsample during saving of the zarr, this is good during testing for visual inspection
shape of the zarr array to create: (715, 1034, 19)
creating zarr at: data\test-series_2\test-series_2.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18


In [19]:
from atlas.io import zarr_array_to_czi

In [20]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

Saving CZI file to: data\test-series_2\test-series_2_ds_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18


In [21]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [22]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

doing full scale alignment, it is recomended to check results before with downscale
shape of the zarr array to create: (6790, 9980, 19)
creating zarr at: data\test-series_2\test-series_2.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18


In [23]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

Saving CZI file to: data\test-series_2\test-series_2_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
